# AdPulse — Advertising Sales Analytics
### Multiple Linear Regression | Prepared by Ashok

**Goal:** estimate product sales from TV, Radio, and Newspaper advertising inputs.

This notebook follows the complete analysis workflow: data inspection → feature selection → train/test split → model fitting → prediction → evaluation → interpretation → visualization → recommendations.

> The dataset is the supplied `Advertising.csv`. Its actual row count is checked in the notebook rather than hard-coded.

## 1. Import the toolkits
The libraries below handle tables, numeric operations, charts, splitting, model training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visual identity for this version: teal, coral, and indigo.
TEAL = "#0f766e"
CORAL = "#fb7185"
INDIGO = "#6366f1"

## 2. Load and inspect the data
Check the first few records, dataset dimensions, data types, missing values, and summary statistics.

In [ ]:
df = pd.read_csv("Advertising.csv")
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

required_columns = ["TV", "Radio", "Newspaper", "Sales"]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

display(df.head())
print("Dataset shape:", df.shape)
print("Missing values per column:")
display(df.isna().sum().to_frame("Missing values"))
display(df.describe().T)

## 3. Define the predictors and target
The three advertising channels are the input features (`X`); `Sales` is the target (`y`).

In [ ]:
features = ["TV", "Radio", "Newspaper"]
target = "Sales"
X = df[features]
y = df[target]

print("Feature columns:", features)
print("Target column:", target)
print("Input shape:", X.shape)
print("Target shape:", y.shape)

## 4. Create training and testing sets
Use 80% of the records for training and 20% for testing. The fixed random state makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Training records: {len(X_train)}")
print(f"Testing records: {len(X_test)}")

## 5. Train the regression model
Multiple Linear Regression estimates the relationship between the three advertising channels and sales.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept: {model.intercept_:.4f}")
for channel, coefficient in zip(features, model.coef_):
    print(f"{channel} coefficient: {coefficient:.4f}")

print("\nEstimated equation:")
print(f"Sales = {model.intercept_:.4f} + ({model.coef_[0]:.4f} × TV) + ({model.coef_[1]:.4f} × Radio) + ({model.coef_[2]:.4f} × Newspaper)")

## 6. Predict sales for unseen records
The model predicts sales for test records that were not used during training.

In [ ]:
y_pred = model.predict(X_test)
prediction_table = pd.DataFrame({
    "Actual sales": y_test.to_numpy(),
    "Predicted sales": y_pred,
    "Absolute error": np.abs(y_test.to_numpy() - y_pred)
})
display(prediction_table.head(10))

## 7. Test a new advertising budget
Example budget: TV = 150, Radio = 20, Newspaper = 30. Change these values to test another scenario.

In [ ]:
sample_budget = pd.DataFrame({"TV": [150], "Radio": [20], "Newspaper": [30]})
sample_prediction = model.predict(sample_budget)[0]
print("Input budget:")
display(sample_budget)
print(f"Estimated sales: {sample_prediction:.2f}")

## 8. Evaluate model quality
MAE and RMSE express typical prediction error in the target's units; MSE squares errors; R² measures the share of target variation explained on the test split.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({"Metric": ["MAE", "MSE", "RMSE", "R²"], "Value": [mae, mse, rmse, r2]})
display(metrics.style.format({"Value": "{:.4f}"}))

## 9. Interpret the coefficients
A coefficient describes the estimated change in sales for a one-unit increase in that input while the other advertising inputs remain fixed. Raw coefficients are not a perfect general-purpose feature-importance comparison when input scales differ.

In [ ]:
coefficient_table = pd.DataFrame({
    "Channel": features,
    "Coefficient": model.coef_,
    "Absolute coefficient": np.abs(model.coef_)
}).sort_values("Absolute coefficient", ascending=False)
display(coefficient_table.style.format({"Coefficient": "{:.4f}", "Absolute coefficient": "{:.4f}"}))
print("Largest absolute coefficient:", coefficient_table.iloc[0]["Channel"])
print("Smallest absolute coefficient:", coefficient_table.iloc[-1]["Channel"])

## 10. Visual check — actual vs predicted sales
Points closer to the diagonal reference line indicate predictions closer to the observed values.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test, y_pred, color=TEAL, edgecolor="white", alpha=0.85, s=65)
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
ax.plot([low, high], [low, high], linestyle="--", color=CORAL, linewidth=2, label="Ideal estimate")
ax.set_xlabel("Actual sales")
ax.set_ylabel("Predicted sales")
ax.set_title("Model check: observed vs estimated sales")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.18)
fig.tight_layout()
plt.show()

## 11. Explore each advertising channel
These scatterplots show the relationship between each advertising input and recorded sales.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
palette = [TEAL, CORAL, INDIGO]
for ax, channel, color in zip(axes, features, palette):
    ax.scatter(df[channel], df[target], color=color, alpha=0.75, edgecolor="white", linewidth=0.3)
    ax.set_xlabel(f"{channel} advertising")
    ax.set_ylabel("Sales")
    ax.set_title(f"{channel} vs sales")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.18)
fig.tight_layout()
plt.show()

## 12. Business and technical recommendations

- Use the model to compare possible budget combinations before making a spending decision.
- Interpret coefficients while keeping the other channels constant; do not treat the result as proof of causation.
- Evaluate additional models such as Random Forest or Gradient Boosting and compare them using the same test split or cross-validation.
- Investigate unusual observations and potential nonlinear relationships before using the model for real decisions.

## 13. Summary
The analysis loaded and checked the data, selected the three advertising channels as predictors, trained Multiple Linear Regression, generated test and sample-budget predictions, measured model performance, interpreted coefficients, and created visual comparisons.

**Prepared by Ashok — AdPulse Sales Analytics.**